In [38]:
import numpy as np
import plotly.graph_objects as go

# Parameters for the geometry
rho = 1.0                # Base radius
d_rho = 0.03              # Radial thickness (∂ρ)
d_theta = np.pi / 10     # Fixed polar angle step (∂θ)
phi_start = np.pi / 5    # Start of our wedge
d_phi = np.pi / 6        # Fixed azimuthal angle step (∂φ)

# Precompute Mesh Connectivity for an 11x11 Grid
# (Using Mesh3d instead of Surface completely prevents Plotly animation x/y update bugs)
GRID_SIZE = 11
I_idx, J_idx, K_idx = [], [], []
for r_idx in range(GRID_SIZE - 1):
    for c_idx in range(GRID_SIZE - 1):
        n0 = r_idx * GRID_SIZE + c_idx
        n1 = n0 + 1
        n2 = (r_idx + 1) * GRID_SIZE + c_idx
        n3 = n2 + 1
        I_idx.extend([n0, n1])
        J_idx.extend([n1, n3])
        K_idx.extend([n2, n2])

# --- 1. Generate the Base Transparent Sphere ---
theta_base, phi_base = np.mgrid[0:np.pi:31j, 0:2*np.pi:31j]
base_x = rho * np.sin(theta_base) * np.cos(phi_base)
base_y = rho * np.sin(theta_base) * np.sin(phi_base)
base_z = rho * np.cos(theta_base)

trace_base_sphere = go.Surface(
    x=base_x, y=base_y, z=base_z,
    surfacecolor=np.ones_like(base_z), # Uniform color
    colorscale=[[0, '#e0f2fe'], [1, '#e0f2fe']], # Light sky blue
    showscale=False,
    opacity=0.25,
    hoverinfo='skip',
    name='Sphere'
)

# --- 2. Generate Axes ---
def create_axis(x, y, z, label):
    return go.Scatter3d(
        mode='lines+text',
        x=[0, x], y=[0, y], z=[0, z],
        text=['', label],
        textposition='top center',
        textfont=dict(size=16, color='black'),
        line=dict(color='#94a3b8', width=4), # Slate-400
        hoverinfo='skip',
        showlegend=False
    )

trace_axis_x = create_axis(1.6, 0, 0, 'x')
trace_axis_y = create_axis(0, 1.6, 0, 'y')
trace_axis_z = create_axis(0, 0, 1.6, 'z')

# --- 3. Generate Wedge Boundaries (Longitudes) ---
def get_longitude(phi):
    t = np.linspace(0, np.pi, 31)
    lx = np.sin(t) * np.cos(phi)
    ly = np.sin(t) * np.sin(phi)
    lz = np.cos(t)
    return lx, ly, lz

long1_x, long1_y, long1_z = get_longitude(phi_start)
long2_x, long2_y, long2_z = get_longitude(phi_start + d_phi)

trace_long1 = go.Scatter3d(
    mode='lines',
    x=long1_x, y=long1_y, z=long1_z,
    line=dict(color='#3b82f6', width=3), # Blue-500
    hoverinfo='skip', showlegend=False
)
trace_long2 = go.Scatter3d(
    mode='lines',
    x=long2_x, y=long2_y, z=long2_z,
    line=dict(color='#3b82f6', width=3),
    hoverinfo='skip', showlegend=False
)

# --- 4. Helper Functions for the 3D Volume Element ---
def get_patch_coords(theta_start, r_val):
    # Generates the 3D coordinates for a curved face, flattened for Mesh3d
    t, p = np.mgrid[theta_start:theta_start+d_theta:11j, phi_start:phi_start+d_phi:11j]
    X = r_val * np.sin(t) * np.cos(p)
    Y = r_val * np.sin(t) * np.sin(p)
    Z = r_val * np.cos(t)
    return X.flatten(), Y.flatten(), Z.flatten()

def get_wedge_lines(theta_start):
    r_out = rho + 0
    
    # Inner Corners
    c1_in = [rho*np.sin(theta_start)*np.cos(phi_start), rho*np.sin(theta_start)*np.sin(phi_start), rho*np.cos(theta_start)]
    c2_in = [rho*np.sin(theta_start)*np.cos(phi_start+d_phi), rho*np.sin(theta_start)*np.sin(phi_start+d_phi), rho*np.cos(theta_start)]
    c3_in = [rho*np.sin(theta_start+d_theta)*np.cos(phi_start), rho*np.sin(theta_start+d_theta)*np.sin(phi_start), rho*np.cos(theta_start+d_theta)]
    c4_in = [rho*np.sin(theta_start+d_theta)*np.cos(phi_start+d_phi), rho*np.sin(theta_start+d_theta)*np.sin(phi_start+d_phi), rho*np.cos(theta_start+d_theta)]

    # Outer Corners
    c1_out = [r_out*np.sin(theta_start)*np.cos(phi_start), r_out*np.sin(theta_start)*np.sin(phi_start), r_out*np.cos(theta_start)]
    c2_out = [r_out*np.sin(theta_start)*np.cos(phi_start+d_phi), r_out*np.sin(theta_start)*np.sin(phi_start+d_phi), r_out*np.cos(theta_start)]
    c3_out = [r_out*np.sin(theta_start+d_theta)*np.cos(phi_start), r_out*np.sin(theta_start+d_theta)*np.sin(phi_start), r_out*np.cos(theta_start+d_theta)]
    c4_out = [r_out*np.sin(theta_start+d_theta)*np.cos(phi_start+d_phi), r_out*np.sin(theta_start+d_theta)*np.sin(phi_start+d_phi), r_out*np.cos(theta_start+d_theta)]

    # Wireframe connecting the 3D block
    lx = [
        0, c1_in[0], None, 0, c2_in[0], None, 0, c3_in[0], None, 0, c4_in[0], None,
        c1_in[0], c2_in[0], c4_in[0], c3_in[0], c1_in[0], None,
        c1_out[0], c2_out[0], c4_out[0], c3_out[0], c1_out[0], None,
        c1_in[0], c1_out[0], None, c2_in[0], c2_out[0], None, c3_in[0], c3_out[0], None, c4_in[0], c4_out[0], None
    ]
    ly = [
        0, c1_in[1], None, 0, c2_in[1], None, 0, c3_in[1], None, 0, c4_in[1], None,
        c1_in[1], c2_in[1], c4_in[1], c3_in[1], c1_in[1], None,
        c1_out[1], c2_out[1], c4_out[1], c3_out[1], c1_out[1], None,
        c1_in[1], c1_out[1], None, c2_in[1], c2_out[1], None, c3_in[1], c3_out[1], None, c4_in[1], c4_out[1], None
    ]
    lz = [
        0, c1_in[2], None, 0, c2_in[2], None, 0, c3_in[2], None, 0, c4_in[2], None,
        c1_in[2], c2_in[2], c4_in[2], c3_in[2], c1_in[2], None,
        c1_out[2], c2_out[2], c4_out[2], c3_out[2], c1_out[2], None,
        c1_in[2], c1_out[2], None, c2_in[2], c2_out[2], None, c3_in[2], c3_out[2], None, c4_in[2], c4_out[2], None
    ]
    return lx, ly, lz

def get_labels(theta_start):
    r_out = rho + d_rho
    
    # Coordinate for ∂ρ (Radial Edge)
    c4_in = np.array([rho*np.sin(theta_start+d_theta)*np.cos(phi_start+d_phi), rho*np.sin(theta_start+d_theta)*np.sin(phi_start+d_phi), rho*np.cos(theta_start+d_theta)])
    c4_out = np.array([r_out*np.sin(theta_start+d_theta)*np.cos(phi_start+d_phi), r_out*np.sin(theta_start+d_theta)*np.sin(phi_start+d_phi), r_out*np.cos(theta_start+d_theta)])
    dr_mid = (c4_in + c4_out) / 2 * 1.0
    
    # Coordinate for ρ ∂θ (Polar Edge)
    c2_out = np.array([r_out*np.sin(theta_start)*np.cos(phi_start+d_phi), r_out*np.sin(theta_start)*np.sin(phi_start+d_phi), r_out*np.cos(theta_start)])
    dtheta_mid = (c2_out + c4_out) / 2 * 1.0
    
    # Coordinate for ρ sin(θ) ∂φ (Azimuthal Edge)
    c3_out = np.array([r_out*np.sin(theta_start+d_theta)*np.cos(phi_start), r_out*np.sin(theta_start+d_theta)*np.sin(phi_start), r_out*np.cos(theta_start+d_theta)])
    dphi_mid = (c3_out + c4_out) / 2 * 1.0
    
    tx = [dr_mid[0], dtheta_mid[0], dphi_mid[0]]
    ty = [dr_mid[1], dtheta_mid[1], dphi_mid[1]]
    tz = [dr_mid[2], dtheta_mid[2], dphi_mid[2]]
    
    return tx, ty, tz

# Initial Patch Data (at theta = 0)
init_px_in, init_py_in, init_pz_in = get_patch_coords(0, rho)
init_px_out, init_py_out, init_pz_out = get_patch_coords(0, rho + d_rho)
init_lx, init_ly, init_lz = get_wedge_lines(0)
init_tx, init_ty, init_tz = get_labels(0)

trace_patch_inner = go.Mesh3d(
    x=init_px_in, y=init_py_in, z=init_pz_in,
    i=I_idx, j=J_idx, k=K_idx,
    color='#F4D363', opacity=0.9, hoverinfo='skip', name='Inner Face'
)

# trace_patch_outer = go.Mesh3d(
#     x=init_px_out, y=init_py_out, z=init_pz_out,
#     i=I_idx, j=J_idx, k=K_idx,
#     color='#F4D363', opacity=0.9, hoverinfo='skip', name='Outer Face'
# )

trace_wedge_lines = go.Scatter3d(
    mode='lines',
    x=init_lx, y=init_ly, z=init_lz,
    line=dict(color='#333741', width=4), # Amber-600
    hoverinfo='skip', showlegend=False
)

trace_labels = go.Scatter3d(
    mode='text',
    x=init_tx, y=init_ty, z=init_tz,
    text=['<b>∂ρ</b>', '<b>ρ ∂θ</b>', '<b>ρ sin(θ) ∂φ</b>'],
    textfont=dict(size=14, color='#333741'), # Dark amber
    showlegend=False, hoverinfo='skip'
)

# Assemble initial figure data
data = [trace_base_sphere, trace_axis_x, trace_axis_y, trace_axis_z, 
        trace_long1, trace_long2, 
        trace_patch_inner, trace_wedge_lines, trace_labels]

# --- 5. Layout Configuration ---
layout = go.Layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=0, r=0, b=60, t=20),
    scene=dict(
        xaxis=dict(visible=False, showgrid=False, zeroline=False),
        yaxis=dict(visible=False, showgrid=False, zeroline=False),
        zaxis=dict(visible=False, showgrid=False, zeroline=False),
        camera=dict(eye=dict(x=1.6, y=1.2, z=0.6)), # Positioned to see the wedge clearly
        aspectratio=dict(x=1, y=1, z=1)
    ),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        x=0.5, y=-0.05,
        xanchor='center', yanchor='top',
        direction='right',
        pad=dict(r=15, l=15, t=10, b=10),
        font=dict(size=14, color='#334155'),
        buttons=[
            dict(
                label='  ▶ Play Animation  ',
                method='animate',
                args=[None, dict(
                    mode='immediate',
                    fromcurrent=True,
                    transition=dict(duration=0), # WebGL surfaces do not transition smoothly
                    frame=dict(duration=60, redraw=True)
                )]
            ),
            dict(
                label='  ❚❚ Pause  ',
                method='animate',
                args=[[None], dict(
                    mode='immediate',
                    transition=dict(duration=0),
                    frame=dict(duration=0, redraw=False)
                )]
            )
        ]
    )]
)

fig = go.Figure(data=data, layout=layout)

# --- 6. Animation Frames ---
frames = []
num_frames = 60
max_theta = np.pi - d_theta

for i in range(num_frames + 1):
    # Calculate a ping-pong theta value
    progress = i / num_frames 
    if progress < 0.5:
        t = progress * 2 * max_theta
    else:
        t = (1 - progress) * 2 * max_theta
    
    px_in, py_in, pz_in = get_patch_coords(t, rho)
    px_out, py_out, pz_out = get_patch_coords(t, rho + d_rho)
    lx, ly, lz = get_wedge_lines(t)
    tx, ty, tz = get_labels(t)

    frames.append(go.Frame(
        name=f'frame_{i}',
        data=[
            go.Mesh3d(x=px_in, y=py_in, z=pz_in, i=I_idx, j=J_idx, k=K_idx),
            go.Scatter3d(x=lx, y=ly, z=lz),
            go.Scatter3d(x=tx, y=ty, z=tz)
        ],
        traces=[6, 7, 8, 9] # Tell Plotly to only update the last 4 dynamic traces
    ))

fig.frames = frames

# Render the plot
if __name__ == '__main__':
    # This will open the animation in your browser
    fig.show()

TypeError: can only concatenate list (not "float") to list